In [ ]:
# %pip install -r requirements.txt

In [ ]:
import numpy as np
import os
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

/Users/sharleneng/anaconda3/envs/bertopic311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Dataset
Load the pre-cleaned arXiv metadata CSV and inspect its structure.
The dataset contains ~20,000 research paper titles and abstracts across 3 domains:
`electrical_engineering`, `economics`, and `biology`.

In [ ]:
# Load your CSV 
data_path = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'data', 'arxiv_cleaned_metadata.csv')
df = pd.read_csv(data_path)

print(f"Total rows: {len(df)}")
print(df.columns)
# print(df[['title', 'abstract', 'broad_subject', 'primary_category']].head(3))

Mounted at /content/drive
Total rows: 19973
Index(['id', 'title', 'abstract', 'published', 'updated', 'authors',
       'primary_category', 'all_categories', 'queried_category',
       'broad_subject', 'title_clean', 'abstract_clean', 'text',
       'abstract_word_count', 'title_word_count', 'base_id', 'version'],
      dtype='object')


In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")

Shape: (19973, 17)

Columns:
['id', 'title', 'abstract', 'published', 'updated', 'authors', 'primary_category', 'all_categories', 'queried_category', 'broad_subject', 'title_clean', 'abstract_clean', 'text', 'abstract_word_count', 'title_word_count', 'base_id', 'version']

Missing values:
id                     0
title                  0
abstract               0
published              0
updated                0
authors                0
primary_category       0
all_categories         0
queried_category       0
broad_subject          0
title_clean            0
abstract_clean         0
text                   0
abstract_word_count    0
title_word_count       0
base_id                1
version                0
dtype: int64


In [ ]:
pd.set_option('display.max_colwidth', 300)

# Look at the key columns side by side
df[['title_clean', 'abstract_clean', 'text', 'broad_subject', 'primary_category']].head(5)

,title_clean,abstract_clean,text,broad_subject,primary_category
0,holographic bound and protein linguistics,the holographic bound in physics constrains the complexity of life. the finite storage capability of information in the observable universe requires the protein linguistics in the evolution of life. we find that the evolution of genetic code determines the variance of amino acid frequencies and ...,holographic bound and protein linguistics the holographic bound in physics constrains the complexity of life. the finite storage capability of information in the observable universe requires the protein linguistics in the evolution of life. we find that the evolution of genetic code determines t...,biology,q-bio.GN
1,mismatch repair error implies chargaff's second parity rule,"chargaff's second parity rule holds empirically for most types of dna that along single strands of dna the base contents are equal for complimentary bases, a = t, g = c. a markov chain model is constructed to track the evolution of any single base position along single strands of genomes whose o...","mismatch repair error implies chargaff's second parity rule chargaff's second parity rule holds empirically for most types of dna that along single strands of dna the base contents are equal for complimentary bases, a = t, g = c. a markov chain model is constructed to track the evolution of any ...",biology,q-bio.GN
2,non-coding dna programs express adaptation and its universal law,significant fraction (98.5% in humans) of most animal genomes is non- coding dark matter. its largely unknown function (1-5) is related to programming (rather than to spontaneous mutations) of accurate adaptation to rapidly changing environment. programmed adaptation to the same universal law fo...,non-coding dna programs express adaptation and its universal law significant fraction (98.5% in humans) of most animal genomes is non- coding dark matter. its largely unknown function (1-5) is related to programming (rather than to spontaneous mutations) of accurate adaptation to rapidly changin...,biology,q-bio.GN
3,chromatin folding in relation to human genome function,"three-dimensional (3d) chromatin structure is closely related to genome function, in particular transcription. however, the folding path of the chromatin fiber in the interphase nucleus is unknown. here, we systematically measured the 3d physical distance between pairwise labeled genomic positio...","chromatin folding in relation to human genome function three-dimensional (3d) chromatin structure is closely related to genome function, in particular transcription. however, the folding path of the chromatin fiber in the interphase nucleus is unknown. here, we systematically measured the 3d phy...",biology,q-bio.GN
4,inverted and mirror repeats in model nucleotide sequences,"we analytically and numerically study the probabilistic properties of inverted and mirror repeats in model sequences of nucleic acids. we consider both perfect and non-perfect repeats, i.e. repeats with mismatches and gaps. the considered sequence models are independent identically distributed (...","inverted and mirror repeats in model nucleotide sequences we analytically and numerically study the probabilistic properties of inverted and mirror repeats in model sequences of nucleic acids. we consider both perfect and non-perfect repeats, i.e. repeats with mismatches and gaps. the considered...",biology,q-bio.GN


In [ ]:
docs = df['text'].tolist()
print(f"Number of documents: {len(docs)}")

print("=== broad_subject distribution ===")
print(df['broad_subject'].value_counts())

print("\n=== primary_category ===")
print(df['primary_category'].value_counts())

Number of documents: 19973
=== broad_subject distribution ===
broad_subject
electrical_engineering    7998
economics                 5995
biology                   5980
Name: count, dtype: int64

=== primary_category ===
primary_category
q-bio.BM    2000
econ.EM     2000
eess.SY     2000
econ.GN     2000
eess.IV     2000
eess.AS     2000
q-bio.NC    1999
eess.SP     1998
econ.TH     1995
q-bio.GN    1981
Name: count, dtype: int64


In [ ]:
df_sample = df.sample(n=19973, random_state=42).reset_index(drop=True)  # use all rows since it's already ~20k
docs = df_sample['text'].tolist()

print(f"Number of documents: {len(docs)}")
print(f"Sample doc preview: {docs[0][:200]}")

Number of documents: 19973
Sample doc preview: change-point detection in time series using mixed integer programming we use cutting-edge mixed integer optimization (mio) methods to develop a framework for detection and estimation of structural bre


## 3. Embedding Model
Load `all-MiniLM-L6-v2`, a lightweight sentence transformer that converts each paper's text into a 384-dimensional vector capturing its semantic meaning.

Why `all-MiniLM-L6-v2`
- fine-tuned specifically for semantic textual similarity tasks using contrastive learning on over 1 billion sentence pairs
-  produces 384-dimensional embeddings with only 6 transformer layers (~22M parameters)
- balances speed-accuracy trade-off, making it practical for large-scale document corpora common in academic literature
- Trained on diverse text sources, the model generalizes well across scientific domains

In [ ]:
# This model is well-suited for scientific/academic text
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Embeddings
Split the dataset by `primary_category`, then encode each domain's documents into embeddings. Results are cached to Google Drive to avoid recomputation on rerun.

In [ ]:
# split by primary_category (10 groups) → find finer subtopics within each subcategory
subcategories = df['primary_category'].unique().tolist()
subcat_dfs = {subcat: df[df['primary_category'] == subcat].reset_index(drop=True)
              for subcat in subcategories}

for subcat, d in subcat_dfs.items():
    print(f"{subcat}: {len(d)} docs")

q-bio.GN: 1981 docs
q-bio.BM: 2000 docs
econ.TH: 1995 docs
econ.EM: 2000 docs
q-bio.NC: 1999 docs
econ.GN: 2000 docs
eess.AS: 2000 docs
eess.IV: 2000 docs
eess.SP: 1998 docs
eess.SY: 2000 docs


In [ ]:
# Generate Embeddings with Caching
EMBED_DIR = '/embeddings'
os.makedirs(EMBED_DIR, exist_ok=True)

# split by primary_category (10 groups) → find finer subtopics within each subcategory
subcat_embeddings = {}

for subcat, d in subcat_dfs.items():
    cache_path = os.path.join(EMBED_DIR, f'embeddings_{subcat.replace("/", "_")}.npy')

    if os.path.exists(cache_path):
        subcat_embeddings[subcat] = np.load(cache_path)
        print(f"{subcat}: loaded from cache {subcat_embeddings[subcat].shape}")
    else:
        docs_subcat = d['text'].tolist()
        embeddings = embedding_model.encode(docs_subcat, show_progress_bar=True)
        np.save(cache_path, embeddings)
        subcat_embeddings[subcat] = embeddings
        print(f"{subcat}: encoded and cached {embeddings.shape}")

q-bio.GN: loaded from cache (1981, 384)
q-bio.BM: loaded from cache (2000, 384)
econ.TH: loaded from cache (1995, 384)
econ.EM: loaded from cache (2000, 384)
q-bio.NC: loaded from cache (1999, 384)
econ.GN: loaded from cache (2000, 384)
eess.AS: loaded from cache (2000, 384)
eess.IV: loaded from cache (2000, 384)
eess.SP: loaded from cache (1998, 384)
eess.SY: loaded from cache (2000, 384)


## 5. Fit BERTopic Model
Fit a separate BERTopic model for each domain. The pipeline runs in 4 stages:
1. **UMAP** — compresses embeddings from 384 to 5 dimensions
2. **HDBSCAN** — clusters similar documents into topics
3. **CountVectorizer** — removes stopwords, extracts unigrams and bigrams
4. **c-TF-IDF** — labels each cluster with its most representative keywords



In [ ]:
# Fit BERTopic Model
NR_TOPICS_SUBCAT = 5  # fewer topics since each subcategory only has ~2000 docs

def make_topic_model_subcat(min_cluster_size=8):
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,         # for tight, dense clusters
        metric='cosine',
        random_state=42
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=3,        # reduces outlier sensitivity
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    vectorizer_model = CountVectorizer(
        stop_words='english',
        min_df=2,
        max_df=2000,          # lowered to match smaller group size
        ngram_range=(1, 2)    # takes unigrams and bigrams
    )
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=True,
        verbose=True
    )

## 6. Hyperparameter Search: Finding Optimal `min_cluster_size`

Before computing final evaluation metrics, we search over candidate
values of `min_cluster_size` to find the configuration that produces
the most coherent topics for each domain.

In [ ]:
min_cluster_sizes = [80, 100, 120, 150, 175, 200]
results = []

for subcat, d in subcat_dfs.items():
    tokenised = [text.split() for text in d['abstract_clean'].tolist()]
    dictionary = Dictionary(tokenised)

    for mcs in min_cluster_sizes:
        model = make_topic_model_subcat(min_cluster_size=mcs)
        topics, _ = model.fit_transform(d['text'].tolist(), subcat_embeddings[subcat])

        topic_info = model.get_topic_info()
        n_topics = len(topic_info[topic_info['Topic'] != -1])
        if n_topics < 2:
            continue

        topics_words = []
        for topic_id in range(n_topics):
            topic = model.get_topic(topic_id)
            if topic:
                topics_words.append([w for w, _ in topic[:10]])

        cv_score   = CoherenceModel(topics=topics_words, texts=tokenised, dictionary=dictionary, coherence='c_v').get_coherence()
        npmi_score = CoherenceModel(topics=topics_words, texts=tokenised, dictionary=dictionary, coherence='c_npmi').get_coherence()

        results.append({
            'subcat': subcat,
            'min_cluster_size': mcs,
            'n_topics': n_topics,
            'coherence_cv':   round(cv_score, 4),
            'coherence_npmi': round(npmi_score, 4),
        })
        print(f"[{subcat}] mcs={mcs} → {n_topics} topics | C_V={cv_score:.4f} | NPMI={npmi_score:.4f}")

results_df = pd.DataFrame(results)

2026-04-03 06:25:03,651 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-03 06:25:18,974 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:25:18,976 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:25:19,178 - BERTopic - Cluster - Completed ✓
2026-04-03 06:25:19,184 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:25:20,125 - BERTopic - Representation - Completed ✓
2026-04-03 06:25:25,620 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[q-bio.GN] mcs=80 → 9 topics | C_V=0.4968 | NPMI=-0.0053


2026-04-03 06:25:33,998 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:25:34,000 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:25:34,229 - BERTopic - Cluster - Completed ✓
2026-04-03 06:25:34,235 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:25:35,177 - BERTopic - Representation - Completed ✓
2026-04-03 06:25:40,267 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[q-bio.GN] mcs=100 → 8 topics | C_V=0.5077 | NPMI=0.0208


2026-04-03 06:25:48,876 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:25:48,878 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:25:49,063 - BERTopic - Cluster - Completed ✓
2026-04-03 06:25:49,068 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:25:50,045 - BERTopic - Representation - Completed ✓
2026-04-03 06:25:54,702 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[q-bio.GN] mcs=120 → 6 topics | C_V=0.4749 | NPMI=0.0188


... (130 outputs left out)

2026-04-03 06:40:48,847 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:40:48,848 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:40:48,951 - BERTopic - Cluster - Completed ✓
2026-04-03 06:40:48,954 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:40:49,651 - BERTopic - Representation - Completed ✓
2026-04-03 06:40:54,437 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[eess.SY] mcs=200 → 4 topics | C_V=0.4045 | NPMI=-0.0270


2026-04-03 06:41:02,226 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:41:02,227 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:41:02,323 - BERTopic - Cluster - Completed ✓
2026-04-03 06:41:02,328 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:41:02,893 - BERTopic - Representation - Completed ✓


[eess.SY] mcs=250 → 3 topics | C_V=0.4072 | NPMI=-0.0178


In [ ]:
# Pick optimal min_cluster_size per subcategory based on combined C_V and NPMI score
# Normalised to 0-1 score
results_df['cv_norm'] = (results_df['coherence_cv'] - results_df['coherence_cv'].min()) / \
                        (results_df['coherence_cv'].max() - results_df['coherence_cv'].min())

results_df['npmi_norm'] = (results_df['coherence_npmi'] - results_df['coherence_npmi'].min()) / \
                          (results_df['coherence_npmi'].max() - results_df['coherence_npmi'].min())

results_df['combined_score'] = (results_df['cv_norm'] + results_df['npmi_norm']) / 2

best = results_df.loc[results_df.groupby('subcat')['combined_score'].idxmax()]

# Extract the optimal min_cluster_size per domain
optimal_mcs = best.set_index('subcat')['min_cluster_size'].to_dict()
print(optimal_mcs)


{'econ.EM': 100, 'econ.GN': 120, 'econ.TH': 100, 'eess.AS': 120, 'eess.IV': 80, 'eess.SP': 80, 'eess.SY': 80, 'q-bio.BM': 175, 'q-bio.GN': 100, 'q-bio.NC': 80}


In [ ]:
subcat_models = {}
subcat_topics = {}

for subcat, d in subcat_dfs.items():
    mcs = optimal_mcs.get(subcat)
    if mcs is None:
        print(f"WARNING: no valid mcs for {subcat}, skipping.")
        continue
    print(f"Refitting {subcat} with min_cluster_size={mcs}")
    model = make_topic_model_subcat(min_cluster_size=mcs)
    topics, _ = model.fit_transform(d['text'].tolist(), subcat_embeddings[subcat])
    subcat_models[subcat] = model
    subcat_topics[subcat] = topics
    print(f"  Topics found: {len(model.get_topic_info()) - 1}")

2026-04-03 06:53:27,297 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Refitting q-bio.GN with min_cluster_size=100


2026-04-03 06:53:35,140 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:53:35,142 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:53:35,246 - BERTopic - Cluster - Completed ✓
2026-04-03 06:53:35,249 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:53:35,938 - BERTopic - Representation - Completed ✓
2026-04-03 06:53:36,496 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


  Topics found: 8
Refitting q-bio.BM with min_cluster_size=175


2026-04-03 06:53:45,659 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:53:45,660 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:53:45,758 - BERTopic - Cluster - Completed ✓
2026-04-03 06:53:45,762 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:53:46,348 - BERTopic - Representation - Completed ✓
2026-04-03 06:53:46,612 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


  Topics found: 4
Refitting econ.TH with min_cluster_size=100


... (12 outputs left out)

2026-04-03 06:54:55,786 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:54:55,787 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:54:55,901 - BERTopic - Cluster - Completed ✓
2026-04-03 06:54:55,904 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:54:56,566 - BERTopic - Representation - Completed ✓
2026-04-03 06:54:56,925 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


  Topics found: 9
Refitting eess.SY with min_cluster_size=80


2026-04-03 06:55:05,860 - BERTopic - Dimensionality - Completed ✓
2026-04-03 06:55:05,861 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-03 06:55:05,967 - BERTopic - Cluster - Completed ✓
2026-04-03 06:55:05,971 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-03 06:55:06,563 - BERTopic - Representation - Completed ✓


  Topics found: 5


## 7. Visualisations
Three interactive Plotly visualisations for topic inspection (best viewed on google colab). Set the `primary category` variable
to switch between the sub-topics of `electrical_engineering`, `economics`, and `biology`.
- **Inter-topic distance map** — shows how distinct or related topics are to each other
- **Keyword bar chart** — top keywords per topic; most useful for human evaluation
- **Document map** — 2D scatter of papers coloured by topic, showing clustering quality
- **Sample documents** — prints 3 paper titles per topic for manual validation

In [ ]:
# Choose which subcategory to visualise
subcat = 'econ.TH'  # change to any: 'econ.EM', 'econ.GN', 'econ.TH', 'eess.SY', 'eess.IV',
                    # 'eess.AS', 'eess.SP', 'q-bio.BM', 'q-bio.NC', 'q-bio.GN'

model_b = subcat_models[subcat]
docs_subcat = subcat_dfs[subcat]['text'].tolist()
embeddings_subcat = subcat_embeddings[subcat]

In [ ]:
# Inter-topic distance map
model_b.visualize_topics()

In [ ]:
# 2D document map
# sample=0.3 since each subcat only has ~2k docs — higher sample is fine
model_b.visualize_documents(docs_subcat, embeddings=embeddings_subcat, sample=0.3)


In [ ]:
# Top keywords per topic
model_b.visualize_barchart()

In [ ]:
# Sample documents per topic
d_b = subcat_dfs[subcat].copy()
d_b['bertopic_topic'] = subcat_topics[subcat]
topic_info_b = model_b.get_topic_info()

pd.set_option('display.max_colwidth', 200)
for _, row in topic_info_b.iterrows():
    if row['Topic'] == -1:
        continue
    topic_docs = d_b[d_b['bertopic_topic'] == row['Topic']][['title', 'bertopic_topic']].head(3)
    print(f"\nTopic {row['Topic']} — {row['Name']} ({row['Count']} docs)")
    print("-" * 60)
    for _, doc_row in topic_docs.iterrows():
        print(f"  • {doc_row['title']}")


Topic 0 — 0_price_model_auction_optimal (369 docs)
------------------------------------------------------------
  • Data Sharing Markets
  • Economic Hysteresis and Its Mathematical Modeling
  • Ex-post implementation with interdependent values

Topic 1 — 1_choice_voting_preference_preferences (250 docs)
------------------------------------------------------------
  • Monotone Comparative Statics in the Calvert-Wittman Model
  • Stable Voting
  • On the probability of the Condorcet Jury Theorem or the Miracle of Aggregation

Topic 2 — 2_learning_information_agents_market (148 docs)
------------------------------------------------------------
  • Uncertain Product Availability in Search Markets
  • Information Acquisition and Diffusion in Markets
  • Truly Costly Search and Word-of-Mouth Communication

Topic 3 — 3_sender_receiver_information_persuasion (118 docs)
------------------------------------------------------------
  • Persuasion and Welfare
  • Persuasion with Ambiguous Receiv

## 8. Compare with Ground Truths Categories




In [ ]:
# Assign topics back to each subcategory DataFrame, then recombine
for subcat in subcategories:
    subcat_dfs[subcat]['bertopic_topic'] = subcat_topics[subcat]

df_results = pd.concat(subcat_dfs.values(), ignore_index=True)

print(f"Total rows: {len(df_results)}")
print(f"\n=== BERTopic topics vs primary_category ===")
for subcat in subcategories:
    print(f"\n--- {subcat} ---")
    d = subcat_dfs[subcat]
    print(d.groupby('bertopic_topic').size().reset_index(name='doc_count'))


Total rows: 19973

=== BERTopic topics vs primary_category ===

--- q-bio.GN ---
   bertopic_topic  doc_count
0              -1        512
1               0        292
2               1        279
3               2        229
4               3        171
5               4        157
6               5        122
7               6        113
8               7        106

--- q-bio.BM ---
   bertopic_topic  doc_count
0              -1        658
1               0        445
2               1        407
3               2        286
4               3        204

--- econ.TH ---
   bertopic_topic  doc_count
0              -1        797
1               0        369
2               1        250
3               2        148
4               3        118
5               4        109
6               5        103
7               6        101

--- econ.EM ---
   bertopic_topic  doc_count
0              -1        281
1               0        711
2               1        607
3               2        2

In [ ]:
for subcat in subcategories:
    subcat_dfs[subcat]['bertopic_topic_subcat'] = subcat_topics[subcat]

df_results_subcat = pd.concat(subcat_dfs.values(), ignore_index=True)

print(f"\n=== BERTopic topics within each subcategory ===")
for subcat in subcategories:
    print(f"\n--- {subcat} ---")
    d = subcat_dfs[subcat]
    topic_info = subcat_models[subcat].get_topic_info()
    for _, row in topic_info.iterrows():
        if row['Topic'] == -1:
            continue
        words = [w for w, _ in subcat_models[subcat].get_topic(row['Topic'])]
        print(f"  Topic {row['Topic']:>3} ({row['Count']:>4} docs): {', '.join(words[:15])}")


=== BERTopic topics within each subcategory ===

--- q-bio.GN ---
  Topic   0 ( 292 docs): reads, sequencing, genome, read, data, assembly, alignment, sequence, reference, mapping
  Topic   1 ( 279 docs): genomes, genome, genes, sequences, dna, evolution, gene, sequence, evolutionary, species
  Topic   2 ( 229 docs): cancer, data, gene, learning, genes, omics, expression, using, based, multi
  Topic   3 ( 171 docs): models, dna, model, language, genomic, sequences, learning, data, sequence, prediction
  Topic   4 ( 157 docs): cell, single cell, single, data, seq, scrna, scrna seq, gene, spatial, cell rna
  Topic   5 ( 122 docs): genetic, gwas, gene, association, traits, studies, genes, risk, disease, analysis
  Topic   6 ( 113 docs): methylation, dna, dna methylation, chromatin, nucleosome, epigenetic, gene, data, genome, cpg
  Topic   7 ( 106 docs): microbial, metagenomic, microbiome, sequencing, metagenome, bacterial, assembly, analysis, data, species

--- q-bio.BM ---
  Topic   0 (

In [ ]:
# outlier rates
print("=== Outlier rate per subcategory ===")
for subcat in subcategories:
    d = subcat_dfs[subcat]
    outliers = (d['bertopic_topic_subcat'] == -1).sum()
    total = len(d)
    print(f"  {subcat}: {outliers}/{total} outliers ({100*outliers/total:.1f}%)")

=== Outlier rate per subcategory ===
  q-bio.GN: 512/1981 outliers (25.8%)
  q-bio.BM: 658/2000 outliers (32.9%)
  econ.TH: 797/1995 outliers (39.9%)
  econ.EM: 281/2000 outliers (14.1%)
  q-bio.NC: 380/1999 outliers (19.0%)
  econ.GN: 486/2000 outliers (24.3%)
  eess.AS: 791/2000 outliers (39.5%)
  eess.IV: 551/2000 outliers (27.6%)
  eess.SP: 433/1998 outliers (21.7%)
  eess.SY: 253/2000 outliers (12.7%)


## 9. Evaluation

### Metrics
- **NPMI Coherence** — measures how often the top words within each topic co-occur in the corpus using Normalized Pointwise Mutual Information (NPMI)
  - Ranges from -1 to 1
  - scores above 0.1 are considered good, above 0.2 strong
- **Topic Coherence (C_V)** — measures how semantically similar the top words within each topic are
  - Ranges from 0 (poor) to 1 (perfect)
  - Computed using gensim, consistent with the LDA evaluation for direct comparison
- **Human Evaluation** — manually inspect top keywords and sample titles per topic to assess whether discovered themes are meaningful and interpretable

### Coherence Benchmark (NPMI)
| Score Range | Interpretation |
|:-----------:|:--------------:|
| < 0.0       | Poor           |
| 0.0 – 0.1   | Moderate       |
| 0.1 – 0.2   | Good           |
| > 0.2       | Strong         |

### Coherence Benchmark (C_V)
| Score Range | Interpretation |
|:-----------:|:--------------:|
| < 0.4       | Poor           |
| 0.4 – 0.55  | Moderate       |
| 0.55 – 0.7  | Good           |
| > 0.7       | Very Good      |



In [ ]:
# Compute NPMI coherence using optimally fitted models
npmi_results = {}

print("\n=== NPMI Coherence Summary ===")

for subcat, model in subcat_models.items():
    d = subcat_dfs[subcat].copy()
    tokenised = [text.split() for text in d['abstract_clean'].tolist()]
    dictionary = Dictionary(tokenised)

    topics_words = []
    for topic_id in range(len(model.get_topic_info()) - 1):
        topic = model.get_topic(topic_id)
        if topic:
            topics_words.append([word for word, _ in topic[:10]])

    if not topics_words:
        print(f"  No valid topics found for {subcat}, skipping.")
        continue

    score = CoherenceModel(
        topics=topics_words,
        texts=tokenised,
        dictionary=dictionary,
        coherence='c_npmi'
    ).get_coherence()

    npmi_results[subcat] = score
    print(f"  {subcat}: {score:.4f}")



=== NPMI Coherence Summary ===
  q-bio.GN: 0.0208
  q-bio.BM: 0.0353
  econ.TH: 0.0386
  econ.EM: 0.0577
  q-bio.NC: 0.0408
  econ.GN: 0.0177
  eess.AS: 0.0333
  eess.IV: 0.0096
  eess.SP: 0.0246
  eess.SY: 0.0069


In [ ]:
# Compute C_V coherence using optimally fitted models
cv_results = {}
print("\n=== C_V Coherence Summary ===")

for subcat, model in subcat_models.items():
    d = subcat_dfs[subcat].copy()
    tokenised = [text.split() for text in d['abstract_clean'].tolist()]
    dictionary = Dictionary(tokenised)

    topics_words = []
    for topic_id in range(len(model.get_topic_info()) - 1):
        topic = model.get_topic(topic_id)
        if topic:
            topics_words.append([word for word, _ in topic[:10]])

    if not topics_words:
        print(f"  No valid topics for {subcat}, skipping.")
        continue

    score = CoherenceModel(
        topics=topics_words,
        texts=tokenised,
        dictionary=dictionary,
        coherence='c_v'
    ).get_coherence()

    cv_results[subcat] = score
    print(f"  {subcat} — C_V: {score:.4f}")


=== C_V Coherence Summary ===
  q-bio.GN — C_V: 0.5077
  q-bio.BM — C_V: 0.5033
  econ.TH — C_V: 0.5617
  econ.EM — C_V: 0.5086
  q-bio.NC — C_V: 0.5022
  econ.GN — C_V: 0.5323
  eess.AS — C_V: 0.4633
  eess.IV — C_V: 0.4650
  eess.SP — C_V: 0.4953
  eess.SY — C_V: 0.4842


In [ ]:
summary = pd.DataFrame({
    'Subcategory': list(cv_results.keys()),
    'Model': 'BERTopic',
    'Num Topics': [
        len(subcat_models[s].get_topic_info()) - 1
        for s in cv_results.keys()
    ],
    'Coherence (C_V)': [round(v, 4) for v in cv_results.values()],
    'Coherence (NPMI)': [
        round(npmi_results.get(s, float('nan')), 4)
        for s in cv_results.keys()
    ]
})

print(summary.to_string(index=False))


Subcategory    Model  Num Topics  Coherence (C_V)  Coherence (NPMI)
   q-bio.GN BERTopic           8           0.5077            0.0208
   q-bio.BM BERTopic           4           0.5033            0.0353
    econ.TH BERTopic           7           0.5617            0.0386
    econ.EM BERTopic           4           0.5086            0.0577
   q-bio.NC BERTopic           5           0.5022            0.0408
    econ.GN BERTopic           6           0.5323            0.0177
    eess.AS BERTopic           4           0.4633            0.0333
    eess.IV BERTopic           8           0.4650            0.0096
    eess.SP BERTopic           9           0.4953            0.0246
    eess.SY BERTopic           5           0.4842            0.0069


### Results
| Subcategory | Num Topics | Coherence (C_V) | Coherence (NPMI) | Interpretation |
|-------------|:----------:|:---------------:|:----------------:|----------------|
| q-bio.GN    | 8          | 0.5077          | 0.0208           | Moderate C_V; positive NPMI indicates meaningful word co-occurrences across genomics sub-themes |
| q-bio.BM    | 4          | 0.5033          | 0.0353           | Fewer topics but above-average NPMI; biomolecular terms cluster coherently despite limited granularity |
| econ.TH     | 7          | 0.5617          | 0.0386           | Best C_V overall; good NPMI too — theoretical economics vocabulary separates well into distinct themes |
| econ.EM     | 4          | 0.5086          | 0.0577           | Highest NPMI across all subcategories; fewest topics but strongest word co-occurrence signal |
| q-bio.NC    | 5          | 0.5022          | 0.0408           | Moderate performance; neuroscience topics are reasonably coherent given broad thematic overlap |
| econ.GN     | 6          | 0.5323          | 0.0177           | Good C_V but lower NPMI; general economics breadth makes tight word co-occurrence harder to achieve |
| eess.AS     | 4          | 0.4633          | 0.0333           | Lowest C_V; audio/speech processing vocabulary may be too specialised for strong semantic separation |
| eess.IV     | 8          | 0.4650          | 0.0096           | Low on both metrics; image/video processing spans too many overlapping technical concepts |
| eess.SP     | 9          | 0.4953          | 0.0246           | Most topics of any subcategory; moderate scores suggest signal processing sub-themes are partially separable |
| eess.SY     | 5          | 0.4842          | 0.0069           | Lowest NPMI; systems engineering jargon is dense and context-dependent, limiting word co-occurrence strength |